# LightGBM — Adult Income Classification

## Dataset Overview

The Adult Income dataset, also known as the Census Income dataset, contains demographic and employment-related information about individuals.

The goal is to predict whether a person's annual income is:

- `<=50K`
- `>50K`

This is a **binary classification problem**.

## Features

The dataset contains a mixture of numerical and categorical features, including:

- Age
- Workclass
- Education
- Education Number
- Marital Status
- Occupation
- Relationship
- Race
- Sex
- Capital Gain
- Capital Loss
- Hours per Week
- Native Country

## Why This Dataset for LightGBM?

This dataset is useful for understanding LightGBM because it contains:

- Numerical and categorical features
- Missing/unknown values
- A relatively large number of observations
- A binary classification target

We will start with the raw dataset, inspect its structure and data-quality issues, perform the necessary preprocessing, and then build a LightGBM classification model.

## Objective

Build a LightGBM model that predicts whether an individual's annual income exceeds $50K.

## Imports

In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer
from lightgbm import LGBMClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv(
    "../dataset/adult.data",
    header=None
)

df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Assigning Column Names

The raw `adult.data` file does not contain column headers, so Pandas initially assigns numerical column names.

We will assign meaningful feature names based on the dataset documentation.

In [3]:
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

df.columns = columns

df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
df.shape

(32561, 15)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [6]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(24)

In [8]:
df.nunique()

age                  73
workclass             9
fnlwgt            21648
education            16
education-num        16
marital-status        7
occupation           15
relationship          6
race                  5
sex                   2
capital-gain        119
capital-loss         92
hours-per-week       94
native-country       42
income                2
dtype: int64

### Numerical features

In [9]:
#don't just check whether there are nulls. Look for values that seem unusual or potentially problematic—especially 
#very large differences between the 75th percentile and maximum.

df.describe() 

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [10]:
df['marital-status'].value_counts()

marital-status
Married-civ-spouse       14976
Never-married            10683
Divorced                  4443
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

In [11]:
df["workclass"].value_counts()

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

In [12]:
df["workclass"].unique()

array([' State-gov', ' Self-emp-not-inc', ' Private', ' Federal-gov',
       ' Local-gov', ' ?', ' Self-emp-inc', ' Without-pay',
       ' Never-worked'], dtype=object)

### Categorical features

In [13]:
categorical_columns = df.select_dtypes(include="object").columns
for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].unique())


workclass
[' State-gov' ' Self-emp-not-inc' ' Private' ' Federal-gov' ' Local-gov'
 ' ?' ' Self-emp-inc' ' Without-pay' ' Never-worked']

education
[' Bachelors' ' HS-grad' ' 11th' ' Masters' ' 9th' ' Some-college'
 ' Assoc-acdm' ' Assoc-voc' ' 7th-8th' ' Doctorate' ' Prof-school'
 ' 5th-6th' ' 10th' ' 1st-4th' ' Preschool' ' 12th']

marital-status
[' Never-married' ' Married-civ-spouse' ' Divorced'
 ' Married-spouse-absent' ' Separated' ' Married-AF-spouse' ' Widowed']

occupation
[' Adm-clerical' ' Exec-managerial' ' Handlers-cleaners' ' Prof-specialty'
 ' Other-service' ' Sales' ' Craft-repair' ' Transport-moving'
 ' Farming-fishing' ' Machine-op-inspct' ' Tech-support' ' ?'
 ' Protective-serv' ' Armed-Forces' ' Priv-house-serv']

relationship
[' Not-in-family' ' Husband' ' Wife' ' Own-child' ' Unmarried'
 ' Other-relative']

race
[' White' ' Black' ' Asian-Pac-Islander' ' Amer-Indian-Eskimo' ' Other']

sex
[' Male' ' Female']

native-country
[' United-States' ' Cuba' ' Jamaica' ' 

### Raw diagnosis so far

| Issue                       | Found?         | Action                                  |
| --------------------------- | -------------- | --------------------------------------- |
| Explicit `NaN`s             | No             | None                                    |
| Duplicate rows              | Yes — 24       | Investigate before deciding             |
| Leading/trailing whitespace | Yes            | Clean                                   |
| Hidden missing values (`?`) | Yes            | Handle                                  |
| Categorical features        | Yes            | Encode appropriately                    |
| Numerical outliers/skew     | Some potential | Investigate, don't automatically remove |
| Binary target               | Yes            | Encode                                  |


In [14]:
(df == " ?").sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

Before we clean it

There are two separate issues:

Whitespace — ' Private' should become 'Private'
Missing marker — ' ?' should become an actual missing value such as NaN

And we should handle them in that order:

strip whitespace → convert ? to NaN → decide how to handle missing values

Why convert to NaN first? Because then Pandas and our preprocessing tools can properly recognize these as missing values.

## Data Cleaning

### Removing Leading and Trailing Whitespace

The categorical values contain unnecessary leading whitespace.

For example, values such as `' Private'` and `' Male'` should be standardized to `'Private'` and `'Male'`.

We will remove leading and trailing whitespace from all categorical columns.

In [15]:
categorical_columns = df.select_dtypes(include="object").columns

df[categorical_columns] = df[categorical_columns].apply(
    lambda column: column.str.strip()
)

### Target distribution

In [16]:
df["income"].value_counts()

income
<=50K    24720
>50K      7841
Name: count, dtype: int64

In [17]:
df["income"].value_counts(normalize=True)

income
<=50K    0.75919
>50K     0.24081
Name: proportion, dtype: float64

So our target is imbalanced, but not extremely so.That means a model could get roughly 76% accuracy simply by predicting <=50K for everyone. So later, accuracy alone won't be enough. We'll pay attention to:

Precision
Recall
F1-score
Confusion matrix

We'll come back to this during evaluation.

### Handling Missing Values

The dataset uses `?` as a placeholder for missing categorical values.

Pandas does not automatically recognize `?` as a missing value, so we will convert these values to `NaN`.

In [18]:
df = df.replace("?", np.nan)

In [19]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
income               0
dtype: int64

Now the important question: what should we do with them?

We have three common options:

Drop rows with missing values
Fill with the mode (most frequent category)
Keep them as a separate category, such as "Unknown"

For this dataset, I would not drop the rows. We have thousands of records with missing categorical information, and dropping them throws away useful data.

And I wouldn't blindly use the mode either, because it would pretend that an unknown workclass/occupation is actually the most common category.

For this project, I'd prefer:

Treat missing categorical values as "Unknown".

So our reasoning is:

Don't invent a category → preserve missingness → let the model potentially learn from it.

In [20]:
# filling the missing values with unknowns

categorical_columns = df.select_dtypes(include="object").columns

df[categorical_columns] = df[categorical_columns].fillna("Unknown")

In [21]:
df.isnull().sum().sum()

np.int64(0)

In [22]:
df.duplicated().sum()

np.int64(24)

### Removing Duplicate Rows

The dataset contains 24 exact duplicate rows.

Since these rows contain identical information across all features and the target, keeping multiple copies can give disproportionate weight to those observations and may cause data leakage if duplicates appear in both the training and test sets.

We will remove the duplicate rows before splitting the data.

In [23]:
df = df.drop_duplicates()

In [24]:
df.duplicated().sum()

np.int64(0)

### Check numerical ranges

In [25]:
# Let's inspect the minimum and maximum of each numerical feature:
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    print(
        f"{column}: min = {df[column].min()}, "
        f"max = {df[column].max()}"
    )

age: min = 17, max = 90
fnlwgt: min = 12285, max = 1484705
education-num: min = 1, max = 16
capital-gain: min = 0, max = 99999
capital-loss: min = 0, max = 4356
hours-per-week: min = 1, max = 99


In [26]:
# Investigate education vs education-num

df.groupby("education")["education-num"].unique()

education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education-num, dtype: object

Every education category maps to exactly one education-num value.
So these two columns are different representations of essentially the same underlying information.

What should we do?

For this LightGBM project, I recommend dropping education and keeping education-num.

Why?

education-num already captures the ordered nature of education:

Preschool < 1st-4th < ... < Masters < Doctorate

Whereas education is the text version of the same information.

Keeping both would give the model redundant information without adding much value.

This is a good example of feature selection based on understanding the data, rather than blindly encoding every column.

### Removing Redundant Features

`education` and `education-num` represent the same underlying education level.

Inspection shows that each `education` category maps to exactly one `education-num` value.

We will retain `education-num` because it provides the same information in an ordered numerical representation and remove the redundant `education` column.

In [27]:
df = df.drop(columns=["education"])

In [28]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education-num', 'marital-status',
       'occupation', 'relationship', 'race', 'sex', 'capital-gain',
       'capital-loss', 'hours-per-week', 'native-country', 'income'],
      dtype='object')

In [29]:
df.shape

(32537, 14)

In [30]:
# identify categorical features
categorical_columns = df.select_dtypes(include="object").columns.tolist()

categorical_columns

['workclass',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country',
 'income']

In [31]:
# Actual categorical features excluding target
categorical_features = [
    column for column in categorical_columns
    if column != "income"
]

categorical_features

['workclass',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

Because with our previous models, we often converted categorical variables into numerical representations such as one-hot encoding.

For this project, we're going to explore a LightGBM-specific capability:

LightGBM can work with categorical features directly when they are appropriately represented.

In [32]:
# categories in each column
for column in categorical_features:
    print(column, "→", df[column].nunique(), "categories")

workclass → 9 categories
marital-status → 7 categories
occupation → 15 categories
relationship → 6 categories
race → 5 categories
sex → 2 categories
native-country → 42 categories


#### Prepare categorical dtype

LightGBM can use categorical features directly when they're represented appropriately.

We'll convert our predictor columns to pandas' category dtype:

In [33]:
for column in categorical_features:
    df[column] = df[column].astype("category")

In [34]:
df[categorical_features].dtypes

workclass         category
marital-status    category
occupation        category
relationship      category
race              category
sex               category
native-country    category
dtype: object

Why category?

Instead of storing each value as a general Python string/object, pandas maintains a set of categories and internally represents the observations using category codes.

More importantly for our project, this gives LightGBM a way to recognize:

"This is a categorical feature; don't treat the category codes as continuous numerical values."

For example, if:

sex:
Female → 0
Male   → 1

we don't want LightGBM to interpret that as:

Male = numerically greater than Female.

It's a categorical relationship, not a continuous numerical scale.

## Feature and Target Separation

The target variable is `income`.

We will separate the predictors (`X`) from the target (`y`) before splitting the data into training and testing sets.

In [35]:
X = df.drop(columns=["income"])
y = df["income"]

In [36]:
X.shape, y.shape

((32537, 13), (32537,))

## Train-Test Split

We will split the dataset into training and testing sets.

Because the target is imbalanced, we will use stratification so that both sets maintain approximately the same class distribution as the original dataset.

We will use 80% of the data for training and 20% for testing.

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

###### Note
For algorithms such as:

Logistic Regression
SVM
KNN

feature scaling can be very important.

For LightGBM, scaling isn't generally necessary because we're building decision trees based on feature split points.

## Baseline LightGBM Model

We will first train a baseline LightGBM classifier using the default model settings.

The purpose of the baseline is to establish an initial performance level before tuning hyperparameters.

In [38]:
import sys
import psutil
import lightgbm

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("psutil:", psutil.__version__)
print("LightGBM:", lightgbm.__version__)

Python: 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]
Python executable: /opt/anaconda3/envs/ml-lightgbm/bin/python
psutil: 7.2.2
LightGBM: 3.3.5


In [39]:
model = LGBMClassifier(
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [40]:
# ignore
import sys
import lightgbm

print("Python:", sys.version.split()[0])
print("LightGBM:", lightgbm.__version__)

Python: 3.11.13
LightGBM: 3.3.5


## Baseline Model Evaluation

We will generate predictions on the unseen test set and evaluate the baseline LightGBM model using the confusion matrix, classification report, and accuracy.

Because the target is imbalanced, we will pay particular attention to precision, recall, and F1-score rather than relying only on accuracy.

In [41]:
y_pred = model.predict(X_test)

In [42]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8787645974185617

Confusion Matrix:
[[4651  289]
 [ 500 1068]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.90      0.94      0.92      4940
        >50K       0.79      0.68      0.73      1568

    accuracy                           0.88      6508
   macro avg       0.84      0.81      0.83      6508
weighted avg       0.88      0.88      0.88      6508



##### so, baseline :


Accuracy: 87.88%
F1 for >50K: 0.73

#### Set up hyperparameter tuning


In [45]:
# create a scorer specifically for >50K: So we explicitly tell f1_score:
# ">50K is the positive class."
# Otherwise, a generic "f1" scorer may assume the positive class is 1.

f1_greater_50k = make_scorer(
    f1_score,
    pos_label=">50K"
)

#### Define our first tuning grid

In [46]:
param_grid = {
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [100, 200]
}

This gives us:

3 × 2 × 2 = 12 combinations

With 5-fold CV:

12 × 5 = 60 model fits. 
That's enough to learn something without creating an unnecessarily huge search.
Also,

Why these three?
Parameter	--> What we're testing
num_leaves	 --> Tree complexity
learning_rate --> How strongly each boosting round contributes
n_estimators --> 	Number of boosting rounds

And there's an important relationship:

Lower learning rate often requires more boosting rounds.

In [47]:
# Run Grid search CV

grid_search = GridSearchCV(
    estimator=LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
    param_grid=param_grid,
    scoring=f1_greater_50k,
    cv=5,
    n_jobs=-1
)

In [48]:
grid_search.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/pyth

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie... verbosity=-1)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200], 'num_leaves': [15, 31, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",make_scorer(f...os_label=>50K)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate

In [49]:
grid_search.best_params_

{'learning_rate': 0.1, 'n_estimators': 100, 'num_leaves': 31}

In [50]:
grid_search.best_score_

np.float64(0.7113828990063448)

In [51]:
# Evaluate the selected model

best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

In [52]:
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))

Accuracy: 0.8787645974185617

Confusion Matrix:
[[4651  289]
 [ 500 1068]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.90      0.94      0.92      4940
        >50K       0.79      0.68      0.73      1568

    accuracy                           0.88      6508
   macro avg       0.84      0.81      0.83      6508
weighted avg       0.88      0.88      0.88      6508



| Metric           | Baseline |  Tuned |
| ---------------- | -------: | -----: |
| Accuracy         |   87.88% | 87.88% |
| `>50K` Precision |     0.79 |   0.79 |
| `>50K` Recall    |     0.68 |   0.68 |
| `>50K` F1        |     0.73 |   0.73 |

So,tuning did not improve our baseline.

Within the hyperparameter ranges we tested, cross-validation selected the baseline configuration as the best-performing configuration for >50K F1, and the resulting test performance was unchanged.

#### Feature importance

In [53]:
feature_importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance

age               582
fnlwgt            541
hours-per-week    352
capital-gain      320
capital-loss      318
education-num     306
occupation        202
workclass          97
native-country     78
sex                68
relationship       64
marital-status     51
race               21
dtype: int32

In [54]:
# Check for overfitting
y_train_pred = best_model.predict(X_train)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy:", accuracy_score(y_test, y_pred_tuned))

Train accuracy: 0.8923124207614583
Test accuracy: 0.8787645974185617


We'll create a separate model specifically to demonstrate early stopping.

But because our current LightGBM version is 3.3.5, we'll use its API style.

Notice - 
Original data
    ↓
Training + Test
    ↓
Training → Training + Validation

The test set remains untouched.

First, create a validation set from the training data:


In [55]:
# Implementing early stopping

X_train_lgb, X_valid, y_train_lgb, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

### Early Stopping

Instead of choosing a small fixed number of boosting rounds, we allow LightGBM to train for up to 500 rounds.

A validation set is used to monitor performance. If the validation score does not improve for a specified number of rounds, training stops early.

In [56]:
early_stop_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [57]:
# fit it
early_stop_model.fit(
    X_train_lgb,
    y_train_lgb,
    eval_set=[(X_valid, y_valid)],
    eval_metric="binary_logloss",
    categorical_feature=categorical_features,
    early_stopping_rounds=20,
    verbose=False
)

/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/sklearn.py:726: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgbm/basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
/opt/anaconda3/envs/ml-lightgbm/lib/python3.11/site-packages/lightgb

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


What we're doing here

We deliberately allow:

up to 500 trees

but tell LightGBM:

If validation performance doesn't improve for 20 consecutive rounds, stop.

In [58]:
early_stop_model.best_iteration_

168

What does that mean?

LightGBM effectively said:

"I don't need all 500 trees. Around tree 168, validation performance was best, and continuing wasn't improving it enough."

500 possible trees
        ↓
validation monitoring
        ↓
best performance at 168
        ↓
training stops after patience is exhausted

#### Evaluating this model with test set

In [59]:
y_pred_early = early_stop_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_early))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_early))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_early))

Accuracy: 0.8747695144437615

Confusion Matrix:
[[4646  294]
 [ 521 1047]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.90      0.94      0.92      4940
        >50K       0.78      0.67      0.72      1568

    accuracy                           0.87      6508
   macro avg       0.84      0.80      0.82      6508
weighted avg       0.87      0.87      0.87      6508



| Metric           |   Baseline | Early stopping |
| ---------------- | ---------: | -------------: |
| Accuracy         | **87.88%** |         87.48% |
| `>50K` Precision |   **0.79** |           0.78 |
| `>50K` Recall    |   **0.68** |           0.67 |
| `>50K` F1        |   **0.73** |           0.72 |

So the baseline remains slightly better on this test set.
And our baseline model is already quite strong:

Accuracy: 87.88%
>50K F1: 0.73

### Understanding the mistakes


                   Predicted
                 <=50K   >50K
Actual <=50K      4651    289
Actual >50K        500   1068

Correct predictions :

4,651 people who actually earn <=50K were correctly classified.

1,068 people who actually earn >50K were correctly classified.

Incorrect predictions :

289 false positives

The model predicted:

>50K

but the actual income was:

<=50K

500 false negatives

The model predicted:

<=50K

but the actual income was:

>50K

The larger error is therefore missing actual >50K cases.

That's exactly why our >50K recall is only:

0.68

In [60]:
results = X_test.copy()

results["actual"] = y_test
results["predicted"] = y_pred_tuned

misclassified = results[
    results["actual"] != results["predicted"]
]

misclassified.head(10)

,age,workclass,fnlwgt,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,actual,predicted
7275,42,State-gov,104663,16,Never-married,Exec-managerial,Not-in-family,White,Female,0,0,40,Italy,>50K,<=50K
455,40,Private,72887,11,Married-civ-spouse,Machine-op-inspct,Husband,Asian-Pac-Islander,Male,0,0,40,United-States,>50K,<=50K
13914,66,Private,205724,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,24,United-States,>50K,<=50K
23852,29,Private,270421,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,50,United-States,>50K,<=50K
667,42,Private,247019,9,Married-civ-spouse,Craft-repair,Husband,Black,Male,0,0,40,United-States,>50K,<=50K
1582,29,Private,200511,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,50,United-States,<=50K,>50K
25212,52,Federal-gov,207841,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,<=50K,>50K
31823,58,Local-gov,294313,10,Married-civ-spouse,Adm-clerical,Wife,Black,Female,0,0,55,United-States,<=50K,>50K
19014,25,Private,122175,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,45,United-States,>50K,<=50K
116,56,Self-emp-inc,303090,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,50,United-States,<=50K,>50K


#### Probability based error analysis

In [61]:
# Get predicted probabilities for the >50K class
y_test_proba = best_model.predict_proba(X_test)[:, 1]

results = X_test.copy()
results["actual"] = y_test
results["predicted"] = y_pred_tuned
results["probability_>50K"] = y_test_proba

misclassified = results[results["actual"] != results["predicted"]]

misclassified[
    ["actual", "predicted", "probability_>50K"]
].head(10)

,actual,predicted,probability_>50K
7275,>50K,<=50K,0.408630
455,>50K,<=50K,0.222253
13914,>50K,<=50K,0.134512
23852,>50K,<=50K,0.441160
667,>50K,<=50K,0.274481
1582,<=50K,>50K,0.653215
25212,<=50K,>50K,0.717892
31823,<=50K,>50K,0.767472
19014,>50K,<=50K,0.452344
116,<=50K,>50K,0.634606


False negatives — actual >50K
Probability of >50K	Interpretation
0.409	borderline
0.222	fairly confident wrong
0.135	very confident wrong
0.441	borderline
0.274	fairly confident wrong
0.452	very borderline

So some people who actually earn >50K look quite similar to the model's <=50K population.

False positives — actual <=50K

You have probabilities:

0.653, 0.718, 0.767, 0.635

These are interesting because they're not borderline. The model was fairly confident that these people were >50K, yet they were actually <=50K.

#### Let's see whether the model's errors are concentrated in one income direction.

In [62]:
misclassified["error_type"] = np.where(
    (misclassified["actual"] == ">50K") &
    (misclassified["predicted"] == "<=50K"),
    "False Negative",
    "False Positive"
)

misclassified["error_type"].value_counts()

/var/folders/zw/chlq4bkj6xz4ckbc4myldh2c0000gn/T/ipykernel_13244/1320585217.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  misclassified["error_type"] = np.where(


error_type
False Negative    500
False Positive    289
Name: count, dtype: int64

#### So the model has a stronger tendency to under-predict >50K than over-predict it.

#### One final check, Let's compare the average confidence for the two error types:

In [63]:
misclassified.groupby("error_type")["probability_>50K"].mean()

error_type
False Negative    0.286464
False Positive    0.646720
Name: probability_>50K, dtype: float64

The interesting part is the asymmetry:

For false negatives, the model predicted only 28.6% probability of >50K on average.
For false positives, it predicted 64.7% probability of >50K on average.

So this isn't simply a model that is uncertain around 50%. In many cases, it makes fairly confident mistakes.

The model achieved good overall accuracy, but error analysis showed 500 false negatives versus 289 false positives. The model tended to under-predict the >50K class. The average predicted probability for false negatives was 0.286, while for false positives it was 0.647, indicating that some misclassifications were confident errors rather than merely borderline cases.